# 00 · Setup and environment check

**Project:** *Is Finance the Frontier of the Exponential Gap? Intangible Capital, Productivity and the Labor Share in U.S. Financial Services, 1997–2024*
**Target journal:** MDPI *FinTech*

Run the notebooks in order:

| Notebook | Purpose |
|---|---|
| `00_setup_and_environment` | Checks packages, API keys (presence only) and folders |
| `01_data_pipeline` | Pulls BEA, BLS, FRED and Census data, builds `data/processed/panel.csv` |
| `02_exploratory_analysis` | Descriptive statistics and figures |
| `03_hypothesis_tests` | H1–H3: exponential growth, structural breaks, panel fixed effects, double machine learning |
| `04_ml_models` | H4: out-of-sample ML forecasting, Diebold–Mariano tests, SHAP |
| `05_robustness` | Alternative measures, samples, placebo and lead tests, BLS cross-checks |

> **Security:** API keys are read from `.env` and are never printed, logged, cached or written to any output. `.env` is listed in `.gitignore`.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "exgap").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

## 1. Packages
Install anything reported missing with `pip install -r requirements.txt`.

In [2]:
import importlib, platform
print("Python", platform.python_version())
required = ["numpy", "pandas", "scipy", "statsmodels", "linearmodels", "sklearn", "lightgbm",
            "shap", "doubleml", "ruptures", "matplotlib", "seaborn", "requests"]
missing = []
for m in required:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:<13} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        missing.append(m)
        print(f"  {m:<13} MISSING")
if missing:
    print("\nInstall the missing packages:  pip install -r requirements.txt")

Python 3.11.0
  numpy         2.4.6
  pandas        3.1.1
  scipy         1.17.1
  statsmodels   0.15.0
  linearmodels  7.1
  sklearn       1.9.1
  lightgbm      4.7.0
  shap          0.51.0
  doubleml      0.11.4
  ruptures      v1.1.10
  matplotlib    3.11.2
  seaborn       0.13.2
  requests      2.34.2


## 2. API keys (presence only — values are never displayed)

In [3]:
from exgap import config
env_file = config.find_env_file()
print(".env file found:", env_file is not None, f"({env_file.parent.name}/.env)" if env_file else "")
for name, ok in config.key_status().items():
    print(f"  {name:<15} {'found' if ok else 'MISSING'}")

.env file found: True (The_Exponential_Age - Project/.env)
  BEA_API_KEY     found
  BLS_API_KEY     found
  FRED_API_KEY    found
  CENSUS_API_KEY  found


The BEA and FRED keys are required. The BLS key is optional (the API works without one at lower limits), and the Census key is optional for the Business Formation Statistics endpoint.

The loader accepts `NAME=value`, `NAME = value` and `NAME: value`, and turns spaces in names into underscores (so `BEA API_KEY` is read as `BEA_API_KEY`).

## 3. Folders and Git safety

In [4]:
print("Project root:", config.ROOT)
for d in [config.RAW_DIR, config.PROCESSED_DIR, config.TABLES_DIR, config.FIGURES_DIR]:
    print("  ", d.relative_to(config.ROOT), "exists:", d.exists())

gi = config.ROOT / ".gitignore"
ignored = gi.read_text().splitlines() if gi.exists() else []
print("\n.env ignored by git:", any(l.strip() in (".env", "*.env", ".env*") for l in ignored))

Project root: C:\Users\91838\Downloads\The_Exponential_Age - Project\fintech_exponential_gap
   data\raw exists: True
   data\processed exists: True
   results\tables exists: True
   results\figures exists: True

.env ignored by git: True
